In [1]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.6
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 2. Análisis de bloques de ocupación
# DESCRIPCIÓN:
#   Filtra automáticamente los archivos semanales que contienen datos
#   de ocupación reales (Occupancy > 0), descartando semanas vacías.
#   Extrae bloques continuos de ocupación y calcula estadísticas
#   ambientales por bloque.
# ===================================================================

import pandas as pd
import numpy as np
import os
from openpyxl.styles import PatternFill, Font
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR  = '.'
OUTPUT_DIR = 'occupancy_analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']


# ==========================================
# FILTRAR ARCHIVOS CON OCUPACIÓN REAL
# ==========================================

def has_real_occupancy(filepath):
    """
    Devuelve True si el archivo semanal contiene al menos un registro
    de ocupación con valor > 0. Descarta semanas sin datos de ocupación.
    """
    df = pd.read_excel(filepath)
    occ_row = df[df['Variable_Measure'] == 'Occupancy']
    if occ_row.empty:
        return False
    date_cols = [c for c in df.columns if c not in metadata_cols]
    values = occ_row[date_cols].values.flatten()
    numeric = pd.to_numeric(values, errors='coerce')
    return bool((numeric > 0).any())


weekly_files = sorted([f for f in os.listdir(INPUT_DIR) if f.startswith('1.6_data_') and f.endswith('.xlsx')])
print(f"Archivos semanales encontrados: {len(weekly_files)}")

files_with_occupancy = []
files_without_occupancy = []

for fname in weekly_files:
    fpath = os.path.join(INPUT_DIR, fname)
    if has_real_occupancy(fpath):
        files_with_occupancy.append(fname)
    else:
        files_without_occupancy.append(fname)

print(f"\n✅ Archivos CON ocupación ({len(files_with_occupancy)}):")
for f in files_with_occupancy:
    print(f"   - {f}")

print(f"\n⛔ Archivos SIN ocupación ({len(files_without_occupancy)}) — se omiten:")
for f in files_without_occupancy:
    print(f"   - {f}")


# ==========================================
# EXTRACCIÓN DE BLOQUES DE OCUPACIÓN
# ==========================================

def extract_occupancy_blocks(df_week, week_name):
    """
    Extrae bloques continuos de ocupación de un DataFrame semanal.
    Un bloque es un período donde la ocupancia tiene el mismo valor.
    """
    occ_row = df_week[df_week['Variable_Measure'] == 'Occupancy']
    if occ_row.empty:
        return []

    occ_row = occ_row.iloc[0]
    date_cols = sorted([c for c in df_week.columns if c not in metadata_cols])

    occupancy_sequence = []
    for col in date_cols:
        val = occ_row[col]
        if pd.notna(val):
            try:
                occupancy_sequence.append({
                    'DateTime': pd.to_datetime(col),
                    'Occupancy': int(val)
                })
            except (ValueError, TypeError):
                continue

    if not occupancy_sequence:
        return []

    # Identificar bloques: períodos donde ocupancia es constante
    blocks = []
    current_block = [occupancy_sequence[0]]
    for i in range(1, len(occupancy_sequence)):
        if occupancy_sequence[i]['Occupancy'] != occupancy_sequence[i-1]['Occupancy']:
            blocks.append(current_block)
            current_block = [occupancy_sequence[i]]
        else:
            current_block.append(occupancy_sequence[i])
    blocks.append(current_block)

    sensor_data = df_week[~df_week['Variable_Measure'].isin(['Occupancy', 'Door_Open', 'Window_Open'])]
    date_cols_dt = [pd.to_datetime(c) for c in date_cols]

    rows = []
    for block in blocks:
        block_start = block[0]['DateTime']
        block_end   = block[-1]['DateTime']
        block_indices = [i for i, dt in enumerate(date_cols_dt) if block_start <= dt <= block_end]

        if not block_indices:
            continue

        row = {
            'Occupancy':    block[0]['Occupancy'],
            'Date':         block_start.date(),
            'Day_of_Week':  block_start.strftime('%A'),
            'Start_Time':   block_start.time(),
            'End_Time':     block_end.time(),
            'Duration_min': round((block_end - block_start).total_seconds() / 60),
            'Week':         week_name
        }

        for _, sensor_row in sensor_data.iterrows():
            values = [float(sensor_row[date_cols[idx]]) for idx in block_indices
                      if pd.notna(sensor_row[date_cols[idx]])]
            if values:
                prefix = f"{sensor_row['Sensor_Location']}_{sensor_row['Variable_Measure']}" \
                         if sensor_row['Sensor_Location'] else sensor_row['Variable_Measure']
                row.update({
                    f'{prefix}_mean': np.mean(values),
                    f'{prefix}_std':  np.std(values) if len(values) > 1 else 0,
                    f'{prefix}_max':  np.max(values),
                    f'{prefix}_min':  np.min(values)
                })
        rows.append(row)

    return rows


# ==========================================
# PROCESAR SOLO ARCHIVOS CON OCUPACIÓN
# ==========================================

all_occupancy_rows = []
for fname in files_with_occupancy:
    print(f"\n   Procesando {fname}...")
    df_week = pd.read_excel(os.path.join(INPUT_DIR, fname))
    week_name = fname.replace('1.6_data_', '').replace('.xlsx', '')
    rows = extract_occupancy_blocks(df_week, week_name)
    all_occupancy_rows.extend(rows)
    print(f"      Bloques encontrados: {len(rows)}")


# ==========================================
# GUARDAR RESULTADOS
# ==========================================

if all_occupancy_rows:
    df_occupancy = pd.DataFrame(all_occupancy_rows)

    first_cols = ['Occupancy', 'Date', 'Day_of_Week', 'Start_Time', 'End_Time', 'Duration_min', 'Week']
    other_cols = sorted([c for c in df_occupancy.columns if c not in first_cols])
    df_occupancy = df_occupancy[first_cols + other_cols].sort_values(['Date', 'Start_Time'])

    file_final = os.path.join(OUTPUT_DIR, '1.6_occupancy_blocks_analysis.xlsx')
    with pd.ExcelWriter(file_final, engine='openpyxl') as writer:
        df_occupancy.to_excel(writer, sheet_name='Occupancy_Blocks', index=False)
        ws = writer.sheets['Occupancy_Blocks']

        # Formatear encabezados
        for cell in ws[1]:
            cell.fill = PatternFill(start_color='4472C4', end_color='4472C4', fill_type='solid')
            cell.font = Font(bold=True, size=11, color='FFFFFF')

        # Colorear filas según nivel de ocupancia
        occ_col = list(df_occupancy.columns).index('Occupancy') + 1
        for row in range(2, ws.max_row + 1):
            val = ws.cell(row=row, column=occ_col).value
            if val is not None:
                if val == 0:
                    color = 'C6EFCE'   # Verde claro  — vacía
                elif val < 10:
                    color = 'FFEB9C'   # Amarillo     — baja
                elif val < 20:
                    color = 'FFD966'   # Naranja      — media
                else:
                    color = 'F4B084'   # Salmón       — alta
                ws.cell(row=row, column=occ_col).fill = PatternFill(
                    start_color=color, end_color=color, fill_type='solid')

    print(f"\nGuardado en: {file_final}")
    print(f"Total de bloques: {len(df_occupancy)}")
    print(f"Niveles de ocupación encontrados: {sorted(df_occupancy['Occupancy'].unique())}")
    print(f"Semanas incluidas: {sorted(df_occupancy['Week'].unique())}")
else:
    print("No se encontraron bloques de ocupación.")

print("\n¡Proceso completado!")

Archivos semanales encontrados: 12

✅ Archivos CON ocupación (6):
   - 1.6_data_February_2026_week_2_days_8-14.xlsx
   - 1.6_data_February_2026_week_3_days_15-21.xlsx
   - 1.6_data_February_2026_week_4_days_22-28.xlsx
   - 1.6_data_March_2026_week_1_days_1-7.xlsx
   - 1.6_data_March_2026_week_2_days_8-14.xlsx
   - 1.6_data_March_2026_week_3_days_15-21.xlsx

⛔ Archivos SIN ocupación (6) — se omiten:
   - 1.6_data_December_2025_week_4_days_22-31.xlsx
   - 1.6_data_February_2026_week_1_days_1-7.xlsx
   - 1.6_data_January_2026_week_1_days_1-7.xlsx
   - 1.6_data_January_2026_week_2_days_8-14.xlsx
   - 1.6_data_January_2026_week_3_days_15-21.xlsx
   - 1.6_data_January_2026_week_4_days_22-31.xlsx

   Procesando 1.6_data_February_2026_week_2_days_8-14.xlsx...
      Bloques encontrados: 11

   Procesando 1.6_data_February_2026_week_3_days_15-21.xlsx...
      Bloques encontrados: 15

   Procesando 1.6_data_February_2026_week_4_days_22-28.xlsx...
      Bloques encontrados: 9

   Procesando 1.6_da